# SASRec 全量训练（团队复现包）

数据目录：`SASRec/data/`。请先从 GitHub Release 准备数据，见 `数据与模型下载.md` 或 `python scripts/download_release_assets.py`。


## 参数策略（全量训练，效果优先）

- 以效果优先为目标：适度增大模型容量、训练轮次与评估负采样数。
- 保留设备自适应：CUDA 使用更激进参数，CPU 使用保守参数，避免 OOM。
- 评估默认使用全量用户（`eval_user_limit=None`），以获得更稳定指标。
- 固定随机种子，保证多次实验可复现、可比较。


In [ ]:
from pathlib import Path
import sys

def resolve_sasrec_dir() -> Path:
    cwd = Path.cwd().resolve()
    for c in (cwd, cwd.parent, cwd / "SASRec", cwd.parent / "SASRec"):
        if (c / "data").is_dir() and (c / "scripts").is_dir():
            return c
    raise FileNotFoundError("Cannot locate SASRec/ (need data/ and scripts/).")

SASREC_DIR = resolve_sasrec_dir()
REPO_ROOT = SASREC_DIR.parent  # optional: monorepo extras (e.g. ItemCF)
if str(SASREC_DIR) not in sys.path:
    sys.path.insert(0, str(SASREC_DIR))
assert (SASREC_DIR / "sasrec_core").is_dir(), "缺少 SASRec/sasrec_core/，请运行 scripts/sync_sasrec_core.py"

cache_dir = SASREC_DIR / "data"
results_dir = SASREC_DIR / "results" / "grid_search"
results_dir.mkdir(parents=True, exist_ok=True)
print("SASREC_DIR:", SASREC_DIR)
print("cache_dir:", cache_dir)


In [ ]:
from __future__ import annotations

import shutil
from pathlib import Path

import pandas as pd
import torch

from sasrec_core import (
    SASRecConfig,
    SASRecEstimator,
    build_memmap_cache,
    load_memmap_meta,
)


split_train_path = cache_dir / "train.parquet"
split_valid_path = cache_dir / "valid.parquet"
split_test_path = cache_dir / "test.parquet"
item_map_path = cache_dir / "item2idx_mapping.parquet"

# 按你的要求：清理首次训练产物后重训
cleanup_old_artifacts = True
old_memmap_dir = cache_dir / "memmap_cache"
old_model_path = cache_dir / "sasrec_full_memmap.pt"

cache_dir.mkdir(parents=True, exist_ok=True)

required_files = [split_train_path, split_valid_path, split_test_path, item_map_path]
missing = [p for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError(f"缺少训练输入文件: {missing}")

if cleanup_old_artifacts:
    if old_memmap_dir.exists():
        shutil.rmtree(old_memmap_dir)
        print("已删除旧 memmap 缓存:", old_memmap_dir)
    if old_model_path.exists():
        old_model_path.unlink()
        print("已删除旧模型文件:", old_model_path)

print("project_root:", project_root)
print("cache_dir:", cache_dir)
print("训练数据:", split_train_path)
print("验证数据:", split_valid_path)
print("测试数据:", split_test_path)
print("映射文件:", item_map_path)


In [ ]:
# 直接使用现成 split 数据，不再从 data.parquet 重新切分
train_rows = int(pd.read_parquet(split_train_path, columns=["user_id"]).shape[0])
valid_rows = int(pd.read_parquet(split_valid_path, columns=["user_id"]).shape[0])
test_rows = int(pd.read_parquet(split_test_path, columns=["user_id"]).shape[0])
map_rows = int(pd.read_parquet(item_map_path, columns=["item_id"]).shape[0])

print("train 用户数:", train_rows)
print("valid 用户数:", valid_rows)
print("test 用户数:", test_rows)
print("item 映射行数:", map_rows)

if not (train_rows == valid_rows == test_rows):
    print("警告: train/valid/test 行数不一致，请确认切分数据是否正确。")

In [ ]:
# 重新构建 memmap 缓存（前面若 cleanup_old_artifacts=True，目录已被删除）
memmap_dir = build_memmap_cache(
    cache_dir=cache_dir,
    overwrite=True,
)
meta = load_memmap_meta(memmap_dir)

item_map_df = pd.read_parquet(item_map_path, columns=["item_id", "item_idx"])
idx2item = dict(zip(item_map_df["item_idx"].astype(int), item_map_df["item_id"].astype(int)))

print("memmap 目录:", memmap_dir)
print("用户数:", meta["num_users"])
print("交互数:", meta["num_interactions"])
print("物品数(itemnum):", meta["itemnum"])

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
is_cuda = device == "cuda"

# 效果优先参数（兼顾设备可运行性）
batch_size = 512 if is_cuda else 256
num_epochs = 8 if is_cuda else 3
hidden_units = 128 if is_cuda else 64
num_blocks = 3 if is_cuda else 2
num_heads = 4 if is_cuda else 2

# 全量评估：None 表示不限制评估用户数量
eval_user_limit = None

eval_num_neg = 200

config = SASRecConfig(
    maxlen=100,
    hidden_units=hidden_units,
    num_blocks=num_blocks,
    num_heads=num_heads,
    dropout_rate=0.2,
    batch_size=batch_size,
    lr=1e-3,
    num_epochs=num_epochs,
    num_workers=0,
    eval_num_neg=eval_num_neg,
    eval_k=10,
    seed=42,
)

print("device:", device)
print("eval_user_limit:", eval_user_limit)
print(config)

est = SASRecEstimator(config=config, device=device)


In [ ]:
# 基于全量 memmap 数据训练
est.fit(
    input_mode="memmap",
    cache_dir=cache_dir,
    memmap_dir=memmap_dir,
    itemnum=int(meta["itemnum"]),
    idx2item=idx2item,
    rebuild_memmap_cache=False,
    eval_user_limit=eval_user_limit,
    verbose=True,
)


In [ ]:
valid_metrics = est.evaluate(mode="valid", eval_user_limit=eval_user_limit)
test_metrics = est.evaluate(mode="test", eval_user_limit=eval_user_limit)

print("验证集指标:", valid_metrics)
print("测试集指标:", test_metrics)

history_df = pd.DataFrame(est.history)
history_df


In [ ]:
model_path = cache_dir / "sasrec_full_memmap.pt"
est.save(model_path, include_data=False)
print("模型已保存:", model_path)

demo_user = int(pd.read_parquet(split_train_path, columns=["user_id"]).iloc[0, 0])
topk = est.recommend(user_idx=demo_user, k=10, chunk_size=20000)
print("示例用户:", demo_user)
print("Top10 推荐:", topk)